# A spectral line: centroid, width, equivalent width and velocity

A reviewed notebook for the Gravitas compute page. It reads one `gravitas.observation/1` spectrum, handed to it as the text `gravitas_input`, and its two settings as the text `gravitas_params`: the line's rest wavelength and the half-width of the window around it. It reads only what the schema promises: the wavelength and flux columns by their roles and units, and the spectrum's medium and frame.

In the window it fits a straight continuum to the outer third on each side, divides it out, and measures the line's depth below it. The centroid is the depth-weighted mean wavelength of the pixels at least half as deep as the line's deepest; the width is measured at half of that depth, between linearly interpolated crossings; the equivalent width is the area of the depth in angstroms. The velocity is the Doppler shift of the centroid from the rest wavelength, `c (centroid / rest - 1)`.

The uncertainties come from adding noise to the spectrum 400 times and measuring again. The noise is the scatter of the continuum pixels about their fit, because this spectrum carries no per-pixel uncertainty. The random generator is seeded, so a run is repeatable.

It returns an `envelope` (the five numbers with units and uncertainties, as a `gravitas.artifact/1`) and a `table` of the continuum-divided window. It draws no figure, so it needs NumPy only: about 9 MB to download, where a notebook that plots needs about 19 MB.

In [ ]:
# Read the input and the two settings.
import json, hashlib
import numpy as np

obs = json.loads(gravitas_input)
params = json.loads(gravitas_params)
assert obs["format"] == "gravitas.observation" and obs["formatVersion"] == 1, "this notebook reads gravitas.observation/1"
assert obs["kind"] == "spectrum", "this notebook reads a spectrum"
REST = float(params.get("rest", 6564.61))
HALF = float(params.get("halfwidth", 60))
assert 1000 <= REST <= 20000, "the rest wavelength is between 1000 and 20000 angstroms"
assert 10 <= HALF <= 300, "the half-width is between 10 and 300 angstroms"
input_digest = hashlib.sha256(gravitas_input.encode("utf-8")).hexdigest()
C_KM_S = 299792.458

xc = next(c for c in obs["columns"] if c["id"] == obs["spectral"]["column"])
yc = next(c for c in obs["columns"] if c["role"] == "value")
assert xc["unit"] == "Angstrom", "the wavelength column must be in angstroms"
num = lambda c: np.array([np.nan if v is None else v for v in c["values"]], float)
w, f = num(xc), num(yc)
masked = np.zeros(len(w), bool)
for m in obs["masks"]:
    masked[m["rows"]] = True
keep = np.isfinite(w) & np.isfinite(f) & ~masked
w, f = w[keep], f[keep]
sel = np.abs(w - REST) <= HALF
x, y = w[sel] - REST, f[sel]
assert len(x) >= 12, "the window holds fewer than 12 pixels"
edge = np.abs(x) >= (2.0 / 3.0) * HALF
assert edge.sum() >= 6, "the window has too few continuum pixels at its edges"
print(f"{len(x)} pixels within {HALF:g} angstroms of {REST:g} ({obs['spectral']['medium']}, {obs['spectral']['frame']})")

In [ ]:
# One measurement: continuum, depth, centroid, width, equivalent width.
def measure(x, y):
    p = np.polyfit(x[edge], y[edge], 1)
    cont = np.polyval(p, x)
    d = 1.0 - y / cont
    # Absorption if the window is mostly below the continuum, emission if above.
    sign = 1.0 if d.sum() >= 0 else -1.0
    d = sign * d
    top = int(np.argmax(d))
    core = d >= 0.5 * d[top]
    centroid = float(np.sum(x[core] * d[core]) / np.sum(d[core]))
    half = 0.5 * d[top]
    lo = top
    while lo > 0 and d[lo] >= half:
        lo -= 1
    hi = top
    while hi < len(d) - 1 and d[hi] >= half:
        hi += 1
    if d[lo] >= half or d[hi] >= half:
        raise ValueError("the line is wider than the window")
    xl = x[lo] + (half - d[lo]) * (x[lo + 1] - x[lo]) / (d[lo + 1] - d[lo])
    xh = x[hi - 1] + (d[hi - 1] - half) * (x[hi] - x[hi - 1]) / (d[hi - 1] - d[hi])
    step = float(np.median(np.diff(x)))
    return {"centroid": centroid, "fwhm": float(xh - xl), "ew": float(np.sum(d) * step), "depth": float(d[top]), "sign": sign}, cont

best, cont = measure(x, y)
print(f"{'absorption' if best['sign'] > 0 else 'emission'} line: centroid {REST + best['centroid']:.2f}, depth {best['depth']:.3f}")

In [ ]:
# Uncertainties: noise from the continuum scatter, added 400 times.
resid = y[edge] / cont[edge] - 1.0
noise = 1.4826 * np.median(np.abs(resid - np.median(resid)))
rng = np.random.default_rng(20261011)
runs = []
for _ in range(400):
    yy = y + rng.normal(0.0, noise, len(y)) * cont
    try:
        m, _ = measure(x, yy)
    except Exception:
        continue
    runs.append([m["centroid"], m["fwhm"], m["ew"], m["depth"]])
runs = np.array(runs)
assert len(runs) >= 300, "too many noisy trials failed to measure the line"
sd = runs.std(axis=0, ddof=1)
centroid = REST + best["centroid"]
velocity = C_KM_S * (centroid / REST - 1.0)
velocity_err = C_KM_S * sd[0] / REST
print(f"centroid {centroid:.2f} +/- {sd[0]:.2f}; velocity {velocity:.0f} +/- {velocity_err:.0f} km/s; {len(runs)} trials")

In [ ]:
# What the notebook returns: an envelope and a table.
sigma = lambda s: {"kind": "sigma", "sigma": float(s), "basis": "data"}
envelope = {
    "format": "gravitas.artifact", "formatVersion": 1,
    "id": f"compute:line-centroid:{input_digest[:16]}",
    "made": {"app": "gravitas"},
    "source": {"kind": "analysis", "id": "compute/line-centroid", "version": "1", "digest": input_digest},
    "provenance": {
        "credit": obs["credit"], "license": obs["license"], "retrieved": obs["retrieved"],
        "citations": [c["text"] for c in obs["citations"]],
        "reductions": obs["reductions"] + [
            f"Window of {HALF:g} angstroms either side of {REST:g}; straight continuum from the outer third of each side.",
            "Centroid: depth-weighted mean over pixels at least half the deepest. Width at half the deepest depth, linear interpolation. Uncertainties from 400 noisy copies (seeded), noise from the continuum scatter.",
        ],
    },
    "quantities": [
        {"id": "line.centroid", "value": centroid, "unit": "Angstrom", "origin": "measured", "uncertainty": sigma(sd[0])},
        {"id": "line.fwhm", "value": best["fwhm"], "unit": "Angstrom", "origin": "measured", "uncertainty": sigma(sd[1])},
        {"id": "line.equivalent-width", "value": best["ew"], "unit": "Angstrom", "origin": "measured", "uncertainty": sigma(sd[2])},
        {"id": "line.depth", "value": best["depth"], "unit": "", "origin": "measured", "uncertainty": sigma(sd[3])},
        {"id": "line.velocity", "value": velocity, "unit": "km/s", "origin": "derived", "uncertainty": sigma(velocity_err)},
    ],
    "warnings": [
        "the velocity is the shift of the centroid from the rest wavelength you gave; it is a line-of-sight velocity relative to the Sun only if the spectrum is heliocentric and the line is not blended",
        f"the pixel step is {float(np.median(np.diff(x))):.2f} angstroms, {C_KM_S * float(np.median(np.diff(x))) / REST:.0f} km/s at the rest wavelength, so the centroid is good to a fraction of a pixel, not better",
        "the uncertainties count the spectrum's noise only: the pixel step, the window you chose and the continuum's shape are not in them",
    ],
}

table = {
    "format": "gravitas.observation", "formatVersion": 1, "kind": "table",
    "id": f"{obs['id']}:window", "title": f"{obs['title']}: continuum-divided window around {REST:g} angstroms",
    "object": obs["object"], "facility": obs["facility"], "origin": "model",
    "source": {"kind": "import", "id": obs["id"], "version": obs["source"].get("version")},
    "credit": obs["credit"], "license": obs["license"], "retrieved": obs["retrieved"],
    "citations": obs["citations"],
    "reductions": envelope["provenance"]["reductions"],
    "axes": {"x": "wavelength", "y": "normalized"},
    "columns": [
        {"id": "wavelength", "name": "wavelength", "unit": "Angstrom", "role": "x", "values": [float(v) for v in x + REST]},
        {"id": "normalized", "name": "flux over continuum", "unit": "", "role": "value", "values": [float(v) for v in y / cont]},
    ],
    "masks": [], "annotations": [],
}

gravitas_output = {"envelope": envelope, "table": table}
print(len(x), "rows returned")